In [33]:
# installing and importing the required libraries
%pip install langchain-ollama
from langchain_ollama import OllamaLLM


Note: you may need to restart the kernel to use updated packages.


In [36]:
m= OllamaLLM(model='tinyllama')

In [9]:
# Implementing Rg for custom Data

In [34]:
%pip install langchain-community langchain-huggingface sentence-transformers openai pypdf
import openai
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


Note: you may need to restart the kernel to use updated packages.


In [35]:
pdf_path = "RAGPaper.pdf"
pdf_reader = PyPDFLoader(pdf_path)
documents = pdf_reader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

In [37]:
embedding = HuggingFaceEmbeddings()
db= FAISS.from_documents(chunks, embedding)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3569.82it/s]


In [44]:


from langchain_classic.chains import ConversationalRetrievalChain
from langchain_core.prompts import PromptTemplate


# 1. Prompt used to rewrite conversational questions
CQP = PromptTemplate.from_template(
"""
Given the following conversation and a follow-up question,
rephrase the follow-up question as a standalone question.

Chat History:
{chat_history}

Follow Up Input:
{question}

Standalone question:
"""
)


# 2. Prompt used to ANSWER from retrieved documents
QA_PROMPT = PromptTemplate.from_template(
"""
Use only the context below to answer the question.

Give a short and direct answer.
Do not repeat the instructions.
Do not summarize the context.
If the answer is not in the context, say "I don't know."

Context:
{context}

Question:
{question}

Answer:
"""
)


qa = ConversationalRetrievalChain.from_llm(
    llm=m,
    retriever=db.as_retriever(search_kwargs={"k": 2}),
    condense_question_prompt=CQP,

    combine_docs_chain_kwargs={
        "prompt": QA_PROMPT
    },

    return_source_documents=True,
    verbose=False
)



In [45]:
chat_history = []

query = "What is RAG?"

result = qa.invoke({
    "question": query,
    "chat_history": chat_history
})

print(result["answer"])

RAG stands for Rapidly Augmented Graphs, a technique for generating sequences from a set of input documents using a topic model. The sequence is represented as a directed acyclic graph (DAG), with each node representing a document and each edge representing a topic-document association. The graph is then marginally marginalized to obtain a probability distribution over all nodes, which is then used to generate the output sequence for each node. The technique is particularly effective in situations where there are multiple possible solutions to a given task, as it allows for efficient and efficient computation of topics given a given document set.
